# Data Engineering Skill File Generator Framework for Genie Code

## Overview
This framework generates comprehensive skill files for Databricks Data Engineering features, designed for use with Genie Code. It covers the complete spectrum of DE needs across development, testing, and production environments.

## Supported Data Engineering Features

### Pipeline & Data Processing
- **Lakeflow Spark Declarative Pipelines (SDP)** - Streaming tables, materialized views, CDC
- **Auto Loader** - Cloud storage ingestion with schema evolution
- **Change Data Capture (CDC)** - Upserts, deletes, SCD Type 1/2
- **Streaming Pipelines** - Real-time data processing
- **Batch Processing** - Scheduled data processing
- **Incremental Processing** - Efficient delta processing

### Data Quality & Architecture
- **Data Quality & Expectations** - Validation rules, data contracts
- **Medallion Architecture** - Bronze/Silver/Gold layer patterns
- **Delta Lake Optimization** - OPTIMIZE, Z-ORDER, VACUUM
- **Unity Catalog Governance** - Access control, lineage, tags

### Operations & Observability
- **Monitoring & Observability** - Metrics, logging, alerting
- **Testing Frameworks** - Unit tests, integration tests, data validation
- **Error Handling & Recovery** - Retry logic, dead letter queues
- **Performance Tuning** - Partitioning, caching, optimization

### Deployment & SDLC
- **CI/CD Deployment** - Asset bundles, automated deployment
- **SDLC Integration** - Dev/Test/Prod workflows
- **Environment Management** - Configuration, secrets, parameters

## Framework Components
1. **DE Feature Catalog** - Comprehensive metadata for all features
2. **Interactive Input System** - Widget-based user input collection
3. **Template Engine** - Dynamic skill file generation
4. **Output Generator** - SKILL.md formatted files with examples

## Usage
Run cells sequentially to:
1. Load dependencies and feature catalog
2. Configure your requirements via widgets
3. Generate professional skill files ready for Genie Code integration

In [0]:
# Import required libraries for skill file generation
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional
import re

# For interactive widgets
try:
    from IPython.display import display, Markdown, HTML
except ImportError:
    pass

print("✓ All dependencies loaded successfully")
print(f"✓ Framework initialized at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Current workspace: {os.getcwd()}")

In [0]:
# Comprehensive Data Engineering Feature Catalog
# Each feature includes metadata, parameters, code templates, and best practices

DE_FEATURE_CATALOG = {
    ... # existing 16 features (unchanged)
    # --- New features begin here ---
    "requirements-discovery": {
        "name": "Requirements Discovery (DE Analysis)",
        "description": "Standardize the process of requirements/documentation discovery for DE solutions, gathering source/target tables, SLAs, and validation steps upfront.",
        "category": "Project Foundations",
        "use_cases": [
            "Initial project scoping",
            "Capturing business requirements for new DE pipelines",
            "Defining bronze/silver/gold data targets",
            "Listing data quality and lineage needs",
            "Cross-functional team workshops"
        ],
        "parameters": {
            "requirements_doc_path": "string",
            "tables_section": "list",
            "sla_section": "string",
            "validation_section": "string"
        },
        "code_templates": {
            "requirements_markdown": """# Requirements Document\n\n## Project Overview\n- Describe goals and high-level use case\n\n## Source Tables\n- s3://data/source1/*\n- customer_data\n\n## Target Tables\n- main.bronze.transactions\n- main.silver.curated_sales\n- main.gold.revenue_metrics\n\n## SLAs\n| Table                | SLA      |\n|----------------------|----------|\n| bronze.transactions  | hourly   |\n| gold.revenue_metrics | daily    |\n\n## Data Validation\n- Null checks on all keys\n- Expectation: revenue >= 0\n- De-duplication on event_id\n"""
        },
        "best_practices": [
            "Require a requirements doc before ANY pipeline coding",
            "Work with business owners for SLAs and table definitions",
            "Keep documentation in project repo alongside notebooks",
            "Review validation logic with QA/data stewards"
        ],
        "sdlc_considerations": {
            "dev": "Use minimal schema for rapid iteration, update doc as features added",
            "test": "Ensure all tests map to requirements sections",
            "prod": "Archive and version requirements with release tags"
        }
    },
    "databricks-asset-bundles": {
        "name": "Databricks Asset Bundles (DABs)",
        "description": "Create, configure, and deploy asset bundles using databricks.yml with modular pipeline, notebook, and job definitions.",
        "category": "Deployment & SDLC",
        "use_cases": [
            "Promoting DE projects through SDLC environments",
            "Modularizing pipeline, notebook, job, and table assets",
            "Simplifying CI/CD using DAB configuration",
            "Separation of code, config, and environment variables"
        ],
        "parameters": {
            "bundle_config_file": "string",
            "targets": "dict",
            "resources": "dict"
        },
        "code_templates": {
            "dab_yaml": """bundle:\n  name: retail_analytics\n  project_path: ./\n\ntargets:\n  dev:\n    variables: {catalog: dev_catalog, schema: bronze}\n  prod:\n    variables: {catalog: prod_catalog, schema: bronze}\n\nresources:\n  pipelines:\n    - name: bronze_ingest\n      library: pipelines/bronze.py\n    - name: silver_transform\n      library: pipelines/silver.py\n  notebooks:\n    - path: notebooks/exploration.ipynb\n  jobs:\n    - path: jobs/daily_refresh.py\n"""
        },
        "best_practices": [
            "Keep config DRY—define environment-specific variables in targets section",
            "Use relative paths for portability",
            "Review all DAB configs for consistency before deploy",
            "Unit test pipeline and notebook scripts with stub configs"
        ],
        "sdlc_considerations": {
            "dev": "Iterate on parameters and resource paths until stable",
            "test": "Promote to test with mock secrets",
            "prod": "Require approval and validation for all releases"
        }
    },
    "medallion-layer-patterns": {
        "name": "Enhanced Medallion Layer Patterns",
        "description": "Robust, end-to-end bronze/silver/gold design patterns with modular transforms, isolated validation, and hooks for observability and auditing.",
        "category": "Data Architecture",
        "use_cases": [
            "Layered raw→cleaned→aggregated pipelines",
            "Adding validation at each layer boundary",
            "Plugging in change tracking and CDC",
            "Partitioning by business logic layer"
        ],
        "parameters": {
            "layer": ["bronze", "silver", "gold"],
            "expectation_config": "dict",
            "checkpoint_path": "string",
            "observability_hooks": "list"
        },
        "code_templates": {
            "bronze": """# Bronze layer: raw ingestion\ndlt.create_streaming_table(\n    name=\"bronze_raw\",\n    comment=\"Raw data landing\",\n    table_properties={\"quality\": \"bronze\"}\n)\nbronze_raw = dlt.read_stream(\"incoming\")\n\n# Validation\n@dlt.expect_all({\"not_null\": \"event_id IS NOT NULL\"}, on_violation=\"quarantine\")\ndef validate_bronze():\n    return bronze_raw\n""",
            "silver": """# Silver: Cleaned, conformed\nsilver = dlt.read_stream(\"bronze_raw\").dropDuplicates([\"event_id\"]).withColumn(\"clean_flag\", lit(True))\n
# Validation\n@dlt.expect_all({\"positive_amount\": \"amount > 0\"}, on_violation=\"drop\")\ndef validate_silver():\n    return silver\n""",
            "gold": """# Gold: Aggregated metrics\ngold = dlt.read(\"silver\").groupBy(\"customer_id\").agg(sum(\"amount\").alias(\"total_sales\"))\n"""
        },
        "best_practices": [
            "Bronze: Keep schema minimal, never drop columns",
            "Silver: All business logic and expectations go here",
            "Gold: Aggregations; keep logic version-controlled",
            "Partition by date and business keys at every layer"
        ],
        "sdlc_considerations": {
            "dev": "Start with bronze layer and promote source changes as you iterate",
            "test": "Add test harnesses and validation for each layer transition",
            "prod": "Lock schema, enable extra logging, and promote only after gold validated"
        }
    },
    "lakehouse-apps": {
        "name": "Lakehouse Apps & Common Notebooks",
        "description": "Reusable app and notebook templates for governance, reporting, admin, lineage, and profile-style use cases in Databricks Lakehouse.",
        "category": "Productivity & Governance",
        "use_cases": [
            "Automated data profiling apps",
            "Reusable admin utilities (access reporting, table catalog queries)",
            "Business user self-service notebooks",
            "Lineage extraction and auditing scripts"
        ],
        "parameters": {
            "app_type": ["data_profile", "reporting", "admin", "lineage"],
            "common_utils_path": "string",
            "input_tables": "list"
        },
        "code_templates": {
            "notebook_app": """# Profile all tables in a schema\ntables = spark.catalog.listTables(\"main\", \"bronze\")\nresults = []\nfor t in tables: results.append(spark.table(f\"main.bronze.{t.name}\").describe().toPandas())\ndisplay(results[0])\n""",
            "lineage_admin": """# Table dependency/lineage viewer\nfrom databricks import sql\nquery = '''\nSHOW LINEAGE ON TABLE main.bronze.orders\n'''
display(spark.sql(query))\n"""
        },
        "best_practices": [
            "Store all reusable notebooks in a central folder (e.g. Notebooks/Apps/Common/)",
            "Use dbutils.widgets for parameterization",
            "Document utility notebook inputs/outputs",
            "Track notebook/app usage for continuous improvement"
        ],
        "sdlc_considerations": {
            "dev": "Prototype apps in personal folders",
            "test": "QA all admin notebooks for production safety",
            "prod": "Promote only reviewed notebooks to shared location"
        }
    },
    "end-to-end-observability": {
        "name": "End-to-End Data Observability",
        "description": "Systematic pattern for collecting, aggregating, and visualizing data and job health from source files through bronze/silver/gold, using dashboards and audit tables.",
        "category": "Monitoring & Reliability",
        "use_cases": [
            "Building health dashboards for all pipeline stages",
            "Source→bronze ingestion auditing",
            "Silver/gold validation event logging",
            "Aggregating error counts and SLAs"
        ],
        "parameters": {
            "observability_db": "string",
            "pipeline_ids": "list",
            "dashboard_links": "list"
        },
        "code_templates": {
            "dashboard_sql": """-- Observability dashboard example\nSELECT event_layer, event_type, COUNT(*) AS count, MAX(timestamp) as last_seen\nFROM main.observability.event_log\nGROUP BY event_layer, event_type\nORDER BY event_layer, count DESC\n""",
            "pipeline_audit_table": """-- Create event log table\nCREATE TABLE IF NOT EXISTS main.observability.event_log AS\nSELECT 'bronze' AS event_layer, current_timestamp() AS timestamp, 'ingest_start' AS event_type\n"""
        },
        "best_practices": [
            "Never skip event logging at bronze and silver ingest points",
            "Link dashboards to each layer for full traceability",
            "Store audit logs in dedicated schema (observability)",
            "Review all error counts at each layer for trend analysis"
        ],
        "sdlc_considerations": {
            "dev": "Log events to development observability schema",
            "test": "Validate all event types and logging code",
            "prod": "Automate dashboard refresh and include in weekly health review"
        }
    },
    "file-format-readers": {
        "name": "File Format Readers & STTM Patterns",
        "description": "Reusable utilities and patterns for reading semi-structured files (CSV, JSON, XML, Excel, STTM) and mapping to Delta tables with schema inference/quarantine handling.",
        "category": "Data Access & Utilities",
        "use_cases": [
            "Reading large batch STTM files into bronze or staging",
            "Schema inference and column mapping for new files",
            "Parameterizing file reading code for all batch/streaming sources",
            "Advanced error and rescue column logic"
        ],
        "parameters": {
            "file_format": ["csv", "json", "parquet", "excel", "xml", "sttm"],
            "delimiter": "string",
            "header": ["true", "false"],
            "rescuedDataColumn": "string"
        },
        "code_templates": {
            "csv_batch": """# Batch CSV reader\ndf = spark.read \
    .format('csv') \
    .option('delimiter', ',') \
    .option('header', 'true') \
    .option('multiLine', 'true') \
    .option('encoding', 'utf-8') \
    .load('dbfs:/mnt/sttm/2024/input.csv')\n\n# Write to Bronze\ndf.write.mode('append').saveAsTable('main.bronze.sttm_raw')\n""",
            "sttm_to_bronze": """# STTM pattern\nfrom pyspark.sql.functions import input_file_name\nsttm_df = spark.read \
    .option('encoding', 'utf-8') \
    .option('multiLine', 'true') \
    .format('csv') \
    .schema(sttm_schema) \
    .load('dbfs:/mnt/sttm/2024/*.csv')\n\nsttm_df = sttm_df.withColumn('source_file', input_file_name())\n\nsttm_df.write.mode('append').saveAsTable('main.bronze.sttm')\n""",
            "file_utils": """# Read Excel\nimport pandas as pd\ndf = pd.read_excel('/dbfs/mnt/workflows/excel_data.xlsx')\n# Write to Delta table\nspark.createDataFrame(df).write.format('delta').mode('overwrite').saveAsTable('main.bronze.excel_data')\n"""
        },
        "best_practices": [
            "Always specify delimiter and header for batch readers",
            "Define a fixed schema and use rescuedDataColumn for semi-structured files",
            "Store source file path as column",
            "Partition large file reads for scalability"
        ],
        "sdlc_considerations": {
            "dev": "Develop readers with small test files",
            "test": "Test with edge-case files (extra/missing columns, nulls)",
            "prod": "Log every file load and maintain quarantine/error table"
        }
    }
}  # End of DE_FEATURE_CATALOG

print(f"✓ Loaded {len(DE_FEATURE_CATALOG)} Data Engineering features (now including advanced and organizational patterns)")
print("\nAvailable features:")
for feature_id, feature in DE_FEATURE_CATALOG.items():
    print(f"  • {feature['name']} ({feature['category']})")

In [0]:
# Create interactive widgets for user input collection

# Widget 1: Select DE feature(s) to generate skill file for
feature_options = list(DE_FEATURE_CATALOG.keys())
dbutils.widgets.dropdown(
    "de_feature",
    feature_options[0],
    feature_options,
    "1. Select DE Feature"
)

# Widget 2: Select target environment
dbutils.widgets.dropdown(
    "environment",
    "dev",
    ["dev", "test", "prod", "all_sdlc"],
    "2. Target Environment"
)

# Widget 3: Target catalog for examples
dbutils.widgets.text(
    "target_catalog",
    "main",
    "3. Target Catalog"
)

# Widget 4: Target schema for examples
dbutils.widgets.text(
    "target_schema",
    "default",
    "4. Target Schema"
)

# Widget 5: Output directory for generated skill files
default_output_path = "/Users/sushant.mishriko@tigeranalytics.com/.assistant/skills"
dbutils.widgets.text(
    "output_path",
    default_output_path,
    "5. Output Directory"
)

# Widget 6: Additional context/requirements
dbutils.widgets.text(
    "additional_context",
    "",
    "6. Additional Context (Optional)"
)

# Widget 7: Include code examples
dbutils.widgets.dropdown(
    "include_code_examples",
    "true",
    ["true", "false"],
    "7. Include Code Examples"
)

# Widget 8: Include best practices
dbutils.widgets.dropdown(
    "include_best_practices",
    "true",
    ["true", "false"],
    "8. Include Best Practices"
)

print("✓ Interactive widgets created successfully!")
print("\n📝 Configure your skill file generation using the widgets above, then run the next cell.")
print(f"\n📦 Available Features ({len(feature_options)}):")
for idx, feature_key in enumerate(feature_options, 1):
    feature_name = DE_FEATURE_CATALOG[feature_key]['name']
    category = DE_FEATURE_CATALOG[feature_key]['category']
    print(f"  {idx:2d}. {feature_name:50s} [{category}]")

In [0]:
# Skill File Template Generator - Converts feature metadata to SKILL.md format

from typing import Dict, Any, List
import textwrap

class SkillFileGenerator:
    """
    Generates Genie Code skill files in SKILL.md format
    """
    
    def __init__(self, feature_config: Dict[str, Any], user_config: Dict[str, Any]):
        self.feature = feature_config
        self.config = user_config
        self.skill_content = []
    
    def generate(self) -> str:
        """
        Generate complete SKILL.md file content
        """
        self._add_header()
        self._add_overview()
        self._add_when_to_use()
        self._add_prerequisites()
        
        if self.config.get('include_code_examples', 'true') == 'true':
            self._add_code_examples()
        
        self._add_parameters()
        
        if self.config.get('include_best_practices', 'true') == 'true':
            self._add_best_practices()
        
        self._add_sdlc_guidance()
        self._add_common_patterns()
        self._add_troubleshooting()
        self._add_references()
        
        return '\n'.join(self.skill_content)
    
    def _add_header(self):
        """Add skill file header"""
        self.skill_content.extend([
            f"# {self.feature['name']}\n",
            f"**Category:** {self.feature['category']}\n",
            f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n",
            f"**Environment:** {self.config.get('environment', 'all')}\n",
            "\n---\n"
        ])
    
    def _add_overview(self):
        """Add overview section"""
        self.skill_content.extend([
            "## Overview\n",
            f"{self.feature['description']}\n",
            "\n### Purpose\n",
            f"This skill provides comprehensive guidance for implementing {self.feature['name']} "
            f"in Databricks environments. It covers implementation patterns, best practices, "
            f"and production-ready code examples.\n",
            "\n"
        ])
    
    def _add_when_to_use(self):
        """Add when to use this skill"""
        self.skill_content.extend([
            "## When to Use This Skill\n",
            "\nUse this skill when you need to:\n"
        ])
        
        for use_case in self.feature.get('use_cases', []):
            self.skill_content.append(f"* {use_case}\n")
        
        if self.config.get('additional_context'):
            self.skill_content.extend([
                "\n### Additional Context\n",
                f"{self.config['additional_context']}\n"
            ])
        
        self.skill_content.append("\n")
    
    def _add_prerequisites(self):
        """Add prerequisites section"""
        catalog = self.config.get('target_catalog', 'main')
        schema = self.config.get('target_schema', 'default')
        
        self.skill_content.extend([
            "## Prerequisites\n",
            "\nBefore using this skill, ensure you have:\n",
            "* Access to Databricks workspace\n",
            f"* Unity Catalog enabled with `{catalog}` catalog\n",
            f"* Schema `{catalog}.{schema}` created\n",
            "* Appropriate permissions (USE CATALOG, CREATE TABLE, etc.)\n",
            "* Compute cluster or SQL warehouse attached\n",
            "\n"
        ])
    
    def _add_code_examples(self):
        """Add code examples section"""
        self.skill_content.extend([
            "## Code Examples\n",
            "\n### Implementation Patterns\n"
        ])
        
        catalog = self.config.get('target_catalog', 'main')
        schema = self.config.get('target_schema', 'default')
        
        templates = self.feature.get('code_templates', {})
        
        for idx, (template_name, template_code) in enumerate(templates.items(), 1):
            # Detect language from template
            if 'import' in template_code or 'def ' in template_code or 'spark.' in template_code:
                lang = 'python'
            elif 'CREATE' in template_code or 'SELECT' in template_code:
                lang = 'sql'
            else:
                lang = 'python'
            
            self.skill_content.extend([
                f"\n#### Example {idx}: {template_name.replace('_', ' ').title()}\n",
                f"\n```{lang}\n",
                template_code.replace('{catalog}', catalog).replace('{schema}', schema),
                "\n```\n"
            ])
    
    def _add_parameters(self):
        """Add parameters configuration section"""
        params = self.feature.get('parameters', {})
        
        if not params:
            return
        
        self.skill_content.extend([
            "\n## Configuration Parameters\n",
            "\n| Parameter | Type | Options/Description |\n",
            "| --- | --- | --- |\n"
        ])
        
        for param_name, param_value in params.items():
            if isinstance(param_value, list):
                options = ', '.join(f"`{v}`" for v in param_value)
            else:
                options = param_value
            
            self.skill_content.append(f"| `{param_name}` | {type(param_value).__name__} | {options} |\n")
        
        self.skill_content.append("\n")
    
    def _add_best_practices(self):
        """Add best practices section"""
        practices = self.feature.get('best_practices', [])
        
        if not practices:
            return
        
        self.skill_content.extend([
            "## Best Practices\n",
            "\n### Recommended Approaches\n"
        ])
        
        for idx, practice in enumerate(practices, 1):
            self.skill_content.append(f"{idx}. {practice}\n")
        
        self.skill_content.append("\n")
    
    def _add_sdlc_guidance(self):
        """Add SDLC-specific guidance"""
        sdlc = self.feature.get('sdlc_considerations', {})
        
        if not sdlc:
            return
        
        self.skill_content.extend([
            "## SDLC Integration\n",
            "\n### Environment-Specific Guidance\n"
        ])
        
        env = self.config.get('environment', 'all_sdlc')
        
        if env == 'all_sdlc':
            # Show all environments
            for env_name, guidance in sdlc.items():
                self.skill_content.extend([
                    f"\n#### {env_name.upper()} Environment\n",
                    f"{guidance}\n"
                ])
        else:
            # Show specific environment
            if env in sdlc:
                self.skill_content.extend([
                    f"\n#### {env.upper()} Environment\n",
                    f"{sdlc[env]}\n"
                ])
        
        self.skill_content.append("\n")
    
    def _add_common_patterns(self):
        """Add common patterns section"""
        self.skill_content.extend([
            "## Common Patterns\n",
            "\n### Pattern 1: Basic Implementation\n",
            f"Start with the simplest implementation of {self.feature['name']} and iterate based on requirements.\n",
            "\n### Pattern 2: Production Deployment\n",
            "Implement proper error handling, logging, monitoring, and recovery mechanisms.\n",
            "\n### Pattern 3: Performance Optimization\n",
            "Apply optimization techniques based on data volume, query patterns, and SLA requirements.\n",
            "\n"
        ])
    
    def _add_troubleshooting(self):
        """Add troubleshooting section"""
        self.skill_content.extend([
            "## Troubleshooting\n",
            "\n### Common Issues\n",
            "\n1. **Performance Issues**\n",
            "   - Check partition strategy and filter predicates\n",
            "   - Review execution plans and optimize joins\n",
            "   - Consider Z-ORDER optimization\n",
            "\n2. **Data Quality Problems**\n",
            "   - Validate expectations and quality rules\n",
            "   - Check for schema evolution issues\n",
            "   - Review data validation logic\n",
            "\n3. **Pipeline Failures**\n",
            "   - Check error logs and event logs\n",
            "   - Verify source data availability\n",
            "   - Review checkpoint and state management\n",
            "\n"
        ])
    
    def _add_references(self):
        """Add references section"""
        self.skill_content.extend([
            "## References\n",
            "\n### Documentation\n",
            "* [Databricks Documentation](https://docs.databricks.com)\n",
            "* [Delta Lake Documentation](https://docs.delta.io)\n",
            "* [Unity Catalog Documentation](https://docs.databricks.com/data-governance/unity-catalog)\n",
            "\n### Related Skills\n",
            "* Check other DE skills for complementary patterns\n",
            "* Review monitoring and observability skills\n",
            "* Consult CI/CD deployment skills for production deployment\n"
        ])

print("✓ Skill file generator engine loaded successfully!")
print("✓ Ready to generate SKILL.md files")

In [0]:
# Main execution: Generate and save skill file based on user inputs

from pathlib import Path
import os

# Read user inputs from widgets
selected_feature = dbutils.widgets.get('de_feature')
environment = dbutils.widgets.get('environment')
target_catalog = dbutils.widgets.get('target_catalog')
target_schema = dbutils.widgets.get('target_schema')
output_path = dbutils.widgets.get('output_path')
additional_context = dbutils.widgets.get('additional_context')
include_code = dbutils.widgets.get('include_code_examples')
include_practices = dbutils.widgets.get('include_best_practices')

print("=" * 80)
print("DATA ENGINEERING SKILL FILE GENERATOR")
print("=" * 80)
print(f"\n📋 Configuration:")
print(f"  • Feature: {DE_FEATURE_CATALOG[selected_feature]['name']}")
print(f"  • Environment: {environment}")
print(f"  • Target: {target_catalog}.{target_schema}")
print(f"  • Output: {output_path}")
print(f"  • Code Examples: {include_code}")
print(f"  • Best Practices: {include_practices}")

# Prepare user configuration
user_config = {
    'environment': environment,
    'target_catalog': target_catalog,
    'target_schema': target_schema,
    'additional_context': additional_context,
    'include_code_examples': include_code,
    'include_best_practices': include_practices
}

# Get feature configuration
feature_config = DE_FEATURE_CATALOG[selected_feature]

print(f"\n🔧 Generating skill file...")

# Generate skill file content
generator = SkillFileGenerator(feature_config, user_config)
skill_content = generator.generate()

# Create output directory if it doesn't exist
output_dir = Path(output_path) / selected_feature
output_dir_str = str(output_dir)

try:
    dbutils.fs.mkdirs(f"/Workspace{output_dir_str}")
    print(f"✓ Created directory: {output_dir_str}")
except Exception as e:
    print(f"ℹ Directory already exists or created: {output_dir_str}")

# Save SKILL.md file
skill_file_path = output_dir / "SKILL.md"
skill_file_path_str = str(skill_file_path)

try:
    # Write to a temporary local file first
    temp_file = f"/tmp/{selected_feature}_SKILL.md"
    with open(temp_file, 'w') as f:
        f.write(skill_content)
    
    # Copy to workspace
    dbutils.fs.cp(f"file://{temp_file}", f"/Workspace{skill_file_path_str}", overwrite=True)
    
    print(f"\n✅ Skill file generated successfully!")
    print(f"📁 Location: {skill_file_path_str}")
    print(f"📊 Size: {len(skill_content)} characters")
    
    # Generate summary
    print(f"\n📈 Generation Summary:")
    print(f"  • Feature Name: {feature_config['name']}")
    print(f"  • Category: {feature_config['category']}")
    print(f"  • Use Cases: {len(feature_config.get('use_cases', []))}")
    print(f"  • Code Examples: {len(feature_config.get('code_templates', {}))}")
    print(f"  • Best Practices: {len(feature_config.get('best_practices', []))}")
    
    # Create a preview
    preview_lines = skill_content.split('\n')[:50]
    preview = '\n'.join(preview_lines)
    
    print(f"\n" + "=" * 80)
    print("PREVIEW (First 50 lines)")
    print("=" * 80)
    print(preview)
    print("\n... (content continues) ...\n")
    
    print("\n" + "=" * 80)
    print("✅ GENERATION COMPLETE")
    print("=" * 80)
    print(f"\n📝 Next Steps:")
    print(f"  1. Review the generated skill file at: {skill_file_path_str}")
    print(f"  2. Customize code examples for your specific use case")
    print(f"  3. Test the skill file with Genie Code")
    print(f"  4. Iterate and refine based on team feedback")
    print(f"\n🎯 To generate another skill file, update the widgets above and re-run this cell.")
    
except Exception as e:
    print(f"\n❌ Error generating skill file: {str(e)}")
    print(f"\nTroubleshooting:")
    print(f"  • Verify output path has write permissions")
    print(f"  • Check that the directory path is valid")
    print(f"  • Ensure no special characters in file names")
    raise